# HM-MAMMQA: leakage-safe full-corpus retrieval + multi-agent multimodal QA

This notebook merges practical HM-RAG-inspired planning, routing, dense retrieval and
graph-assisted reranking with MAMMQA-style modality experts, cross-modal synthesis,
verification, one controlled retry, aggregation and abstention. Gold annotations are
isolated until inference completes. Web retrieval is disabled for benchmark runs.

**Audit of the previous notebook.** Reused: robust JSONL loading, table-to-markdown
conversion, one-time 4-bit Qwen loading, PIL/Qwen vision-message construction, and
appendable JSONL outputs. Replaced: metadata-ID context lookup, Wikipedia image
substitution, unavailable image anchors, XML/regex answer protocols, first-answer-only
scoring, first-N sampling, the mislabeled ToT routine, and non-resumable experiment loops.

In [ ]:
# Cell 1 - environment diagnostics (never replace Kaggle's torch/CUDA stack)
import importlib.util, platform, subprocess, sys
print("Python:", sys.version)
try:
    import torch
    print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))
except Exception as e: print("PyTorch diagnostic failed:", e)

In [ ]:
# Cell 2 - install only missing, released packages
REQUIRED = {
    "transformers": "transformers>=4.49,<5", "accelerate": "accelerate>=1,<2",
    "bitsandbytes": "bitsandbytes>=0.45,<1", "qwen_vl_utils": "qwen-vl-utils>=0.0.8",
    "sentence_transformers": "sentence-transformers>=3,<6", "faiss": "faiss-cpu>=1.8,<2",
    "networkx": "networkx>=3,<4", "pandas": "pandas>=2,<3", "tabulate": "tabulate>=0.9,<1",
}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else: print("All optional dependencies are already installed.")

In [ ]:
# Cell 3 - imports, configuration and seed
import ast, csv, gc, hashlib, json, math, os, pickle, random, re, string, time, zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Sequence, Tuple
import faiss, networkx as nx, numpy as np, pandas as pd, torch
from PIL import Image
from sentence_transformers import SentenceTransformer

SEED=42; RUN_MODE="smoke"; MODEL_NAME="Qwen/Qwen2.5-VL-7B-Instruct"
EMBED_MODEL="BAAI/bge-small-en-v1.5"; CLIP_MODEL="openai/clip-vit-base-patch32"
TEXT_TOP_K=5; TABLE_TOP_K=3; IMAGE_TOP_K=3; IMAGE_CANDIDATES=40
MAX_RETRIEVAL_ROUNDS=2; ENABLE_GRAPH=True; ENABLE_WEB=False
IMAGE_RETRIEVAL_MODE="hybrid"  # hybrid | clip_full | metadata_only(debug only)
AGGREGATOR_SEES_QUESTION=True; TABLE_CHUNK_ROWS=50; LARGE_TABLE_ROWS=120
MAX_INPUT_TOKENS=1792; MAX_LLM_NEW_TOKENS=192
RUN_ABLATIONS=False; CLIP_DEVICE="cpu"  # avoids competing with model-parallel Qwen
OUTPUT_ROOT=Path("/kaggle/working/hm_mammqa")
DIRS={n:OUTPUT_ROOT/n for n in ["cache","indexes","images","predictions","retrieval_logs","ablations","metrics"]}
for p in DIRS.values(): p.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert not ENABLE_WEB, "Official MMQA experiments require ENABLE_WEB=False"

In [ ]:
# Cell 4 - confirmed MMQA paths with unambiguous fallback
DATA_ROOT=Path("/kaggle/input/datasets/adibraian/multimodalqa-dataset")
NAMES={
 "train":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_train.jsonl",
 "dev":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_dev.jsonl",
 "test":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_test.jsonl",
 "texts":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_texts.jsonl",
 "tables":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_tables.jsonl",
 "images":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_images.jsonl"}
FOLDERS={k:f"MMQA_{k}.jsonl" for k in NAMES}
def resolve_dataset_file(key:str)->Path:
    primary=DATA_ROOT/FOLDERS[key]/NAMES[key]
    if primary.is_file(): return primary
    roots=[DATA_ROOT,Path("/kaggle/input")]
    matches=sorted({p.resolve() for root in roots if root.exists() for p in root.rglob(NAMES[key])})
    if len(matches)!=1: raise FileNotFoundError(f"{key}: expected one {NAMES[key]}, found {matches}")
    return matches[0]
PATHS={k:resolve_dataset_file(k) for k in NAMES}
for k,p in PATHS.items(): print(f"{k:7s} {p}")

In [ ]:
# Cell 5 - JSONL loading
def load_jsonl(path:Path)->List[dict]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for no,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except json.JSONDecodeError as e: raise ValueError(f"{path}:{no}: {e}") from e
    return out
dev_raw=load_jsonl(PATHS["dev"]); texts_raw=load_jsonl(PATHS["texts"])
tables_raw=load_jsonl(PATHS["tables"]); images_raw=load_jsonl(PATHS["images"])
print(len(dev_raw),len(texts_raw),len(tables_raw),len(images_raw))

In [ ]:
# Cell 6 - schema inspection and hard inference/evaluation separation
for name,row in [("question",dev_raw[0]),("text",texts_raw[0]),("table",tables_raw[0]),("image",images_raw[0])]:
    print(name,{k:type(v).__name__ for k,v in row.items()})
@dataclass(frozen=True)
class QueryInput: qid:str; question:str
@dataclass(frozen=True)
class EvaluationOnly:
    gold_answers:List[str]; gold_support_ids:List[str]; gold_support_modalities:Dict[str,str]
    metadata_type:str="unknown"; gold_modalities:List[str]=field(default_factory=list)
    intermediate_answers:Any=None
FORBIDDEN={"answers","supporting_context","metadata","table_id","text_doc_ids","image_doc_ids",
           "modalities","type","intermediate_answers","gold_answers","gold_support_ids"}
def assert_safe_query(q:QueryInput):
    assert isinstance(q,QueryInput) and set(asdict(q))=={"qid","question"}
    assert not (set(asdict(q)) & FORBIDDEN)
def query_only(row:dict)->QueryInput:
    return QueryInput(str(row.get("qid",row.get("id",""))),str(row["question"]))
print("LEAKAGE CHECK PASSED")

In [ ]:
# Cell 7 - official image discovery, ZIP extraction and pixel validation (URLs never used)
def safe_extract(zp:Path,dst:Path):
    with zipfile.ZipFile(zp) as z:
        root=dst.resolve()
        for member in z.infolist():
            target=(dst/member.filename).resolve()
            if root not in target.parents and target!=root: raise ValueError(f"Unsafe ZIP member: {member.filename}")
        z.extractall(dst)
def discover_image_paths(records:List[dict])->Dict[str,Path]:
    expected={Path(str(r.get("path",""))).name for r in records if r.get("path")}
    mounts=Path("/kaggle/input"); extract=DIRS["images"]/"official"
    named=[] if not mounts.exists() else [p for p in mounts.rglob("final_dataset_images") if p.is_dir()]
    zips=[] if not mounts.exists() else [p for p in mounts.rglob("*.zip") if re.search(r"(mmqa|multimodal).*image|image.*(mmqa|multimodal)",p.name,re.I)]
    valid_zips=[]
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zh:
                if any(Path(n).name in expected for n in zh.namelist()): valid_zips.append(z)
        except zipfile.BadZipFile: pass
    if not named and valid_zips:
        if len(valid_zips)>1: raise RuntimeError(f"Ambiguous official image ZIPs: {valid_zips}")
        marker=extract/".complete"
        if not marker.exists():
            extract.mkdir(parents=True,exist_ok=True); safe_extract(valid_zips[0],extract); marker.touch()
        named=[extract]
    roots=named or ([mounts] if mounts.exists() else [])
    by_name={}
    for root in roots:
        for p in root.rglob("*"):
            if p.is_file() and p.name in expected:
                if p.name in by_name and by_name[p.name]!=p: raise RuntimeError(f"Ambiguous image filename {p.name}")
                by_name[p.name]=p
    return {str(r["id"]):by_name[Path(str(r.get("path",""))).name] for r in records if Path(str(r.get("path",""))).name in by_name}
IMAGE_PATHS=discover_image_paths(images_raw)
ok=[]
for p in list(IMAGE_PATHS.values())[:5]:
    try:
        with Image.open(p) as im: im.verify()
        ok.append(p)
    except Exception as e: raise RuntimeError(f"Invalid official image {p}: {e}")
print(f"total image metadata records={len(images_raw):,}; resolved={len(IMAGE_PATHS):,}; missing={len(images_raw)-len(IMAGE_PATHS):,}; resolution={100*len(IMAGE_PATHS)/max(1,len(images_raw)):.2f}%")
if not IMAGE_PATHS: print("WARNING: official pixels unavailable; image retrieval evaluation is disabled. Text/table development only.")
else: print("PIL pixel checks passed:",len(ok))
FULL_IMAGE_BENCHMARK=bool(IMAGE_PATHS) and IMAGE_RETRIEVAL_MODE in {"hybrid","clip_full"}

In [ ]:
# Cell 8 - corpus normalization and table representations
def cell_text(c:Any)->str:
    if isinstance(c,dict): return str(c.get("text",c.get("value","")))
    return str(c)
def table_parts(row:dict)->Tuple[List[str],List[List[str]]]:
    t=row.get("table",row); header=t.get("header",[]); rows=t.get("table_rows",t.get("rows",[]))
    heads=[str(x.get("column_name",x.get("text",""))) if isinstance(x,dict) else str(x) for x in header]
    return heads,[[cell_text(c) for c in rr] for rr in rows]
def markdown_table(heads,rows)->str:
    width=max([len(heads),*(len(r) for r in rows)] or [0]); heads=(heads+[f"col_{i}" for i in range(len(heads),width)])[:width]
    clean=lambda s:str(s).replace("|","\\|").replace("\n"," ")
    return "\n".join(["| "+" | ".join(map(clean,heads))+" |","| "+" | ".join(["---"]*width)+" |",*["| "+" | ".join(map(clean,r+[""]*(width-len(r))))+" |" for r in rows]])
def normalize_corpus():
    docs=[]
    for r in texts_raw:
        did=str(r["id"]); title=str(r.get("title","")); body=str(r.get("text",r.get("paragraph","")))
        docs.append({"doc_id":did,"parent_doc_id":did,"modality":"text","title":title,"retrieval_text":f"{title}\n{body}","raw_content":body,"image_path":None})
    for r in tables_raw:
        did=str(r["id"]); title=str(r.get("title",r.get("name",""))); h,rows=table_parts(r); raw=markdown_table(h,rows)
        chunks=[rows] if len(rows)<=LARGE_TABLE_ROWS else [rows[i:i+TABLE_CHUNK_ROWS] for i in range(0,len(rows),TABLE_CHUNK_ROWS)]
        for i,ch in enumerate(chunks):
            cid=did if len(chunks)==1 else f"{did}::rows_{i*TABLE_CHUNK_ROWS}-{i*TABLE_CHUNK_ROWS+len(ch)-1}"
            linear=" ; ".join([f"{k}: {v}" for rr in ch for k,v in zip(h,rr)])
            docs.append({"doc_id":cid,"parent_doc_id":did,"modality":"table","title":title,"retrieval_text":f"{title}\ncolumns: {' | '.join(h)}\n{linear}","raw_content":raw,"image_path":None,"source_record":r})
    for r in images_raw:
        did=str(r["id"]); title=str(r.get("title",""))
        docs.append({"doc_id":did,"parent_doc_id":did,"modality":"image","title":title,"retrieval_text":title,"raw_content":{"path":r.get("path"),"url":r.get("url")},"image_path":str(IMAGE_PATHS[did]) if did in IMAGE_PATHS else None})
    return docs
CORPUS=normalize_corpus(); BY_MODALITY={m:[d for d in CORPUS if d["modality"]==m] for m in ["text","table","image"]}
DOC_LOOKUP={(d["modality"],d["doc_id"]):d for d in CORPUS}; PARENT_LOOKUP={(d["modality"],d["parent_doc_id"]):d for d in CORPUS}
print({m:len(v) for m,v in BY_MODALITY.items()})

In [ ]:
# Cells 9-10 - cached normalized FAISS indexes for text and tables
def fingerprint(records):
    h=hashlib.sha256(); h.update(EMBED_MODEL.encode())
    for d in records: h.update((d["doc_id"]+"\0"+d["retrieval_text"]).encode("utf-8"))
    return h.hexdigest()
class DenseIndex:
    def __init__(self,modality,records,encoder): self.modality,self.records,self.encoder=modality,records,encoder; self.index=None
    def build_or_load(self):
        ip=DIRS["indexes"]/f"{self.modality}_faiss.index"; mp=DIRS["indexes"]/f"{self.modality}_doc_mapping.json"; ep=DIRS["cache"]/f"{self.modality}_embeddings.npy"
        fp=fingerprint(self.records)
        if ip.exists() and mp.exists() and ep.exists() and json.loads(mp.read_text())["fingerprint"]==fp: self.index=faiss.read_index(str(ip)); return
        emb=self.encoder.encode([d["retrieval_text"] for d in self.records],batch_size=128,normalize_embeddings=True,show_progress_bar=True).astype("float32")
        np.save(ep,emb); self.index=faiss.IndexFlatIP(emb.shape[1]); self.index.add(emb); faiss.write_index(self.index,str(ip))
        mp.write_text(json.dumps({"fingerprint":fp,"ids":[d["doc_id"] for d in self.records]}))
    def search(self,query,k,subquery=""):
        q=self.encoder.encode([query],normalize_embeddings=True).astype("float32"); scores,idx=self.index.search(q,min(k,len(self.records)))
        return [{**self.records[i],"dense_score":float(s),"graph_score":0.0,"final_score":float(s),"subqueries":[subquery or query]} for s,i in zip(scores[0],idx[0]) if i>=0]
encoder=SentenceTransformer(EMBED_MODEL,device="cuda:0" if torch.cuda.is_available() else "cpu")
TEXT_INDEX=DenseIndex("text",BY_MODALITY["text"],encoder); TEXT_INDEX.build_or_load()
TABLE_INDEX=DenseIndex("table",BY_MODALITY["table"],encoder); TABLE_INDEX.build_or_load()
IMAGE_TITLE_INDEX=DenseIndex("image_titles",BY_MODALITY["image"],encoder); IMAGE_TITLE_INDEX.build_or_load()

In [ ]:
# Cell 11 - official-image CLIP reranking with persistent lazy/full cache
class ClipReranker:
    def __init__(self): self.model=None; self.processor=None; self.cache={}; self.path=DIRS["cache"]/"clip_embeddings.pkl"
    @staticmethod
    def unwrap_features(output):
        # transformers versions differ: older CLIP returns Tensor; newer releases return BaseModelOutputWithPooling.
        if isinstance(output,torch.Tensor): return output
        pooled=getattr(output,"pooler_output",None)
        if isinstance(pooled,torch.Tensor): return pooled
        if isinstance(output,(tuple,list)):
            tensors=[x for x in output if isinstance(x,torch.Tensor) and x.ndim==2]
            if tensors:return tensors[-1]
        raise TypeError(f"Unsupported CLIP feature output: {type(output).__name__}")
    def load(self):
        from transformers import CLIPModel,CLIPProcessor
        self.processor=CLIPProcessor.from_pretrained(CLIP_MODEL,use_fast=False); self.model=CLIPModel.from_pretrained(CLIP_MODEL).eval().to(CLIP_DEVICE)
        if self.path.exists(): self.cache=pickle.loads(self.path.read_bytes())
    def encode_missing(self,docs):
        if self.model is None: self.load()
        missing=[d for d in docs if d["image_path"] and d["doc_id"] not in self.cache]
        for start in range(0,len(missing),32):
            batch=missing[start:start+32]; ims=[]
            for d in batch:
                with Image.open(d["image_path"]) as im:
                    if im.mode=="P" and "transparency" in im.info: im=im.convert("RGBA")
                    ims.append(im.convert("RGB"))
            x=self.processor(images=ims,return_tensors="pt").to(self.model.device)
            with torch.inference_mode(): z=self.unwrap_features(self.model.get_image_features(**x)); z=z/z.norm(dim=-1,keepdim=True)
            self.cache.update({d["doc_id"]:e.cpu().numpy().astype("float32") for d,e in zip(batch,z)})
        if missing: self.path.write_bytes(pickle.dumps(self.cache,pickle.HIGHEST_PROTOCOL))
    def rerank(self,query,docs,k):
        docs=[d for d in docs if d.get("image_path")]; self.encode_missing(docs)
        if not docs:return []
        text_limit=int(getattr(self.model.config.text_config,"max_position_embeddings",77))
        x=self.processor(text=[query],return_tensors="pt",padding=True,truncation=True,max_length=text_limit).to(self.model.device)
        assert x["input_ids"].shape[-1]<=text_limit
        with torch.inference_mode(): q=self.unwrap_features(self.model.get_text_features(**x)); q=q/q.norm(dim=-1,keepdim=True)
        q=q[0].cpu().numpy(); out=[]
        for d in docs:
            score=float(np.dot(q,self.cache[d["doc_id"]])); out.append({**d,"clip_score":score,"final_score":score})
        return sorted(out,key=lambda x:x["final_score"],reverse=True)[:k]
CLIP=ClipReranker()
def retrieve_images(query,k=IMAGE_TOP_K):
    if not IMAGE_PATHS:return []
    pool=IMAGE_TITLE_INDEX.search(query,len(BY_MODALITY["image"]) if IMAGE_RETRIEVAL_MODE=="clip_full" else IMAGE_CANDIDATES,query)
    pool=[d for d in pool if d.get("image_path")]
    if IMAGE_RETRIEVAL_MODE=="metadata_only": return pool[:k]
    return CLIP.rerank(query,pool,k)

In [ ]:
# Cell 12 - lightweight reproducible graph and reciprocal-rank fusion
def norm_entity(x): return re.sub(r"\s+"," ",str(x).strip().lower())
def build_graph():
    path=DIRS["cache"]/"mmqa_graph.pkl"; fp=fingerprint(CORPUS)
    if path.exists():
        payload=pickle.loads(path.read_bytes())
        if payload["fingerprint"]==fp:return payload["graph"]
    g=nx.Graph()
    seen_table_records=set()
    for d in CORPUS:
        dn=f'd::{d["modality"]}::{d["parent_doc_id"]}'; g.add_node(dn,kind="document",doc_id=d["parent_doc_id"],modality=d["modality"])
        title=norm_entity(d["title"])
        if title:g.add_edge(dn,"e::"+title,kind="title")
        if d["modality"]=="table" and d["parent_doc_id"] not in seen_table_records:
            seen_table_records.add(d["parent_doc_id"])
            _,rows=table_parts(d.get("source_record",{}))
            for row in rows:
                ents=[]
                for value in row:
                    v=norm_entity(value)
                    if 1<len(v)<=100 and not v.replace(".","").isdigit(): ents.append("e::"+v)
                ents=list(dict.fromkeys(ents))[:20]
                for e in ents:g.add_edge(dn,e,kind="cell")
                for a,b in zip(ents,ents[1:]):g.add_edge(a,b,kind="row")
    path.write_bytes(pickle.dumps({"fingerprint":fp,"graph":g},pickle.HIGHEST_PROTOCOL)); return g
GRAPH=build_graph() if ENABLE_GRAPH else None
class WebRetriever:
    """Extension point only; web evidence is forbidden in official MMQA experiments."""
    def search(self,query:str,k:int=5):
        if not ENABLE_WEB: raise RuntimeError("Web retrieval is disabled for reproducibility and leakage control")
        raise NotImplementedError("Provide a versioned, auditable backend before enabling non-benchmark experiments")
def graph_rrf(items:List[dict],modality:str,k:int,rrf_k=60):
    if not GRAPH or not items:return items[:k]
    dense_rank={d["parent_doc_id"]:i+1 for i,d in enumerate(items)}; graph_rank={}; scores={}
    for seed in items[:10]:
        n=f'd::{modality}::{seed["parent_doc_id"]}'
        if n not in GRAPH:continue
        lengths=nx.single_source_shortest_path_length(GRAPH,n,cutoff=2)
        for node,hop in lengths.items():
            if node.startswith("d::") and GRAPH.nodes[node].get("modality")==modality:
                did=GRAPH.nodes[node]["doc_id"]; scores[did]=scores.get(did,0)+1/(hop+1)
    for i,(did,_) in enumerate(sorted(scores.items(),key=lambda x:x[1],reverse=True),1):graph_rank[did]=i
    ids=set(dense_rank)|set(graph_rank); out=[]
    for did in ids:
        rec=PARENT_LOOKUP.get((modality,did));
        if not rec:continue
        ds=next((x.get("dense_score",0) for x in items if x["parent_doc_id"]==did),0.0); gs=scores.get(did,0.0)
        final=(1/(rrf_k+dense_rank[did]) if did in dense_rank else 0)+(1/(rrf_k+graph_rank[did]) if did in graph_rank else 0)
        out.append({**rec,"dense_score":ds,"graph_score":gs,"final_score":final,"subqueries":list({q for x in items if x["parent_doc_id"]==did for q in x.get("subqueries",[])})})
    return sorted(out,key=lambda x:x["final_score"],reverse=True)[:k]

In [ ]:
# Cell 13 - retrieval sanity helpers (gold comparison occurs only after retrieval)
def fallback_plan(question):
    relational=bool(re.search(r"\b(where|which|whose|that|who)\b.*\b(where|which|what|when|color|year)\b",question,re.I))
    return {"needs_decomposition":False,"subqueries":[question],"needed_modalities":["text","table","image"],"needs_relational_search":relational}
for row in dev_raw[:3]:
    q=query_only(row); assert_safe_query(q)
    hits=TEXT_INDEX.search(q.question,3)+TABLE_INDEX.search(q.question,3)
    retrieved_ids=[h["parent_doc_id"] for h in hits]
    # Only now, after retrieval, inspect annotations for this diagnostic comparison.
    support_blob=json.dumps(row.get("supporting_context",[]),ensure_ascii=False)
    diagnostic_gold=[str(r["id"]) for r in texts_raw[:0]+tables_raw[:0]]  # type anchor; intentionally empty
    diagnostic_gold=[str(r["id"]) for r in texts_raw if str(r["id"]) in support_blob][:10]+[str(r["id"]) for r in tables_raw if str(r["id"]) in support_blob][:10]
    print(q.qid,"retrieved=",retrieved_ids,"gold_hits=",sorted(set(retrieved_ids)&set(diagnostic_gold)))

In [ ]:
# Cell 14 - unload retrieval encoders before loading Qwen
del encoder; TEXT_INDEX.encoder=TABLE_INDEX.encoder=IMAGE_TITLE_INDEX.encoder=None
if CLIP.model is not None: CLIP.model.to("cpu")
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Cell 15 - one quantized Qwen2.5-VL load
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
processor=AutoProcessor.from_pretrained(MODEL_NAME,min_pixels=256*28*28,max_pixels=512*28*28)
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
model=Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map="auto",low_cpu_mem_usage=True).eval()

In [ ]:
# Cell 16 - structured inference and conservative JSON parsing
def parse_json_object(text:str)->dict:
    candidates=[text]
    candidates += re.findall(r"```(?:json)?\s*(\{.*?\})\s*```",text,re.S|re.I)
    start=text.find("{"); end=text.rfind("}")
    if start>=0 and end>start:candidates.append(text[start:end+1])
    for c in candidates:
        try:
            parsed=json.loads(c)
            if isinstance(parsed,dict):return parsed
            if isinstance(parsed,(str,int,float,bool)):
                value=str(parsed)
                return {"value":value,"candidate_answer":value,"best_answer":value,"answer":value}
        except Exception:pass
    return {}
def llm_json(system:str,payload:dict,image_paths:Optional[List[str]]=None,max_new_tokens=MAX_LLM_NEW_TOKENS)->dict:
    content=[]
    for p in image_paths or []:
        if p and Path(p).is_file():content.append({"type":"image","image":f"file://{p}"})
    content.append({"type":"text","text":json.dumps(payload,ensure_ascii=False)})
    messages=[{"role":"system","content":system},{"role":"user","content":content}]
    prompt=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    imgs,vids=process_vision_info(messages)
    def prepare(limit):
        kwargs={"text":[prompt],"return_tensors":"pt","padding":True,"truncation":True,"max_length":limit}
        if imgs:kwargs.update(images=imgs,videos=vids)
        return processor(**kwargs).to(model.device)
    inputs=prepare(MAX_INPUT_TOKENS)
    try:
        with torch.inference_mode(): ids=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,temperature=None,top_p=None,top_k=None)
    except torch.OutOfMemoryError:
        del inputs; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        inputs=prepare(1280)
        with torch.inference_mode(): ids=model.generate(**inputs,max_new_tokens=min(max_new_tokens,128),do_sample=False,temperature=None,top_p=None,top_k=None)
    raw=processor.batch_decode(ids[:,inputs.input_ids.shape[1]:],skip_special_tokens=True)[0].strip()
    obj=parse_json_object(raw); obj["_raw"]=raw; return obj
def clip01(x,default=0.0):
    try:return max(0.0,min(1.0,float(x)))
    except:return default

In [ ]:
# Cell 17 - question-only planner/decomposer
PLANNER_PROMPT='''Plan corpus retrieval using only the question. Return one JSON object with needs_decomposition:boolean, subqueries:string[], needed_modalities chosen from text/table/image, needs_relational_search:boolean. Never assume hidden metadata. Keep at most 3 subqueries.'''
def plan_query(q:QueryInput)->dict:
    assert_safe_query(q); p=llm_json(PLANNER_PROMPT,{"question":q.question})
    mods=[m for m in p.get("needed_modalities",[]) if m in {"text","table","image"}]
    subs=[str(x).strip() for x in p.get("subqueries",[]) if str(x).strip()][:3]
    if not mods:mods=["text","table","image"]
    if not subs:subs=[q.question]
    return {"needs_decomposition":bool(p.get("needs_decomposition",len(subs)>1)),"subqueries":subs,"needed_modalities":mods,"needs_relational_search":bool(p.get("needs_relational_search",False))}

In [ ]:
# Cell 18 - retrieval router, subquery provenance and optional graph RRF
QUERY_ENCODER=None
def ensure_encoder():
    global QUERY_ENCODER
    # Kept on CPU and loaded once so it can coexist with model-parallel Qwen on the T4s.
    if QUERY_ENCODER is None: QUERY_ENCODER=SentenceTransformer(EMBED_MODEL,device="cpu")
    TEXT_INDEX.encoder=TABLE_INDEX.encoder=IMAGE_TITLE_INDEX.encoder=QUERY_ENCODER
    return QUERY_ENCODER
def merge_ranked(groups,k):
    best={}
    for group in groups:
        for rank,d in enumerate(group,1):
            key=d["parent_doc_id"]; x=best.setdefault(key,{**d,"final_score":0.0,"subqueries":[]})
            x["final_score"]+=1/(60+rank); x["dense_score"]=max(x.get("dense_score",-1e9),d.get("dense_score",-1e9)); x["subqueries"]=list(dict.fromkeys(x["subqueries"]+d.get("subqueries",[])))
    return sorted(best.values(),key=lambda x:x["final_score"],reverse=True)[:k]
def retrieve_evidence(q:QueryInput,plan:dict,use_graph=True,fixed_all=False)->Dict[str,List[dict]]:
    assert_safe_query(q); enc=ensure_encoder(); mods=["text","table","image"] if fixed_all else plan["needed_modalities"]
    queries=[q.question] if fixed_all else plan["subqueries"]; out={m:[] for m in ["text","table","image"]}
    for m in mods:
        groups=[]
        for sq in queries:
            if m=="text":groups.append(TEXT_INDEX.search(sq,max(TEXT_TOP_K,10),sq))
            elif m=="table":groups.append(TABLE_INDEX.search(sq,max(TABLE_TOP_K,10),sq))
            elif m=="image":groups.append(retrieve_images(sq,IMAGE_TOP_K))
        k={"text":TEXT_TOP_K,"table":TABLE_TOP_K,"image":IMAGE_TOP_K}[m]; merged=merge_ranked(groups,max(k,10))
        out[m]=graph_rrf(merged,m,k) if use_graph and plan.get("needs_relational_search") else merged[:k]
    return out

In [ ]:
# Cell 19 - MAMMQA modality experts over retrieved evidence only
EXPERT_PROMPT='''You are a modality evidence expert. Use only supplied retrieved evidence. Return JSON: modality, evidence_summary, candidate_answer (or INSUFFICIENT), confidence 0..1, evidence_ids. IDs must be copied from supplied evidence. No hidden reasoning or external knowledge.'''
def compact_evidence(items,content_chars=1200):
    def bounded(x):
        if isinstance(x,str):return x[:content_chars]
        return x
    return [{"doc_id":d["parent_doc_id"],"title":d["title"],"content":bounded(d["raw_content"]),"retrieval_score":d.get("final_score",0)} for d in items]
def run_modality_experts(q:QueryInput,evidence:Dict[str,List[dict]])->Dict[str,dict]:
    assert_safe_query(q); out={}
    for mod,items in evidence.items():
        available=[d for d in items if mod!="image" or d.get("image_path")]
        if not available:continue
        payload={"question":q.question,"modality":mod,"evidence":compact_evidence(available,1200)}
        paths=[d["image_path"] for d in available] if mod=="image" else None
        x=llm_json(EXPERT_PROMPT,payload,paths); allowed={d["parent_doc_id"] for d in available}
        x.update(modality=mod,confidence=clip01(x.get("confidence")),evidence_ids=[i for i in map(str,x.get("evidence_ids",[])) if i in allowed])
        if not x.get("candidate_answer"):x["candidate_answer"]="INSUFFICIENT"
        out[mod]=x
    return out

In [ ]:
# Cell 20 - active-anchor cross-modal synthesis
SYNTH_PROMPT='''Synthesize only the supplied expert/evidence records, anchored on the named modality. Return JSON: candidate_answer, evidence_summary, supporting_doc_ids, confidence 0..1, status supported|insufficient|conflict. Use INSUFFICIENT when needed. No chain-of-thought.'''
def run_cross_modal_agents(q:QueryInput,experts:Dict[str,dict],evidence:Dict[str,List[dict]])->Dict[str,dict]:
    assert_safe_query(q); out={}; allowed={d["parent_doc_id"] for xs in evidence.values() for d in xs}
    for anchor,a in experts.items():
        x=llm_json(SYNTH_PROMPT,{"question":q.question,"anchor_modality":anchor,"anchor_expert":a,"all_experts":experts,"available_evidence":{m:compact_evidence(v,500) for m,v in evidence.items() if v}})
        x["supporting_doc_ids"]=[i for i in map(str,x.get("supporting_doc_ids",[])) if i in allowed]; x["confidence"]=clip01(x.get("confidence")); out[anchor]=x
    return out

In [ ]:
# Cell 21 - semantic evidence/claim verifier
VERIFY_PROMPT='''Verify candidates against supplied retrieved evidence only. Return JSON: status supported|conflict|insufficient, best_answer, confidence 0..1, supporting_doc_ids, contradicting_doc_ids, missing_information, retry_modality text|table|image|none, retry_query. Do not use answer overlap or outside knowledge.'''
def verify_answer(q:QueryInput,synth:Dict[str,dict],evidence:Dict[str,List[dict]])->dict:
    assert_safe_query(q); allowed={d["parent_doc_id"] for xs in evidence.values() for d in xs}
    x=llm_json(VERIFY_PROMPT,{"question":q.question,"candidates":synth,"retrieved_evidence":{m:compact_evidence(v,600) for m,v in evidence.items()}})
    for key in ["supporting_doc_ids","contradicting_doc_ids"]:x[key]=[i for i in map(str,x.get(key,[])) if i in allowed]
    x["confidence"]=clip01(x.get("confidence")); x["retry_modality"]=x.get("retry_modality","none") if x.get("retry_modality") in {"text","table","image","none"} else "none"
    return x

In [ ]:
# Cells 22-24 - one retry, aggregation and complete proposed pipeline
AGG_PROMPT='''Choose the answer supported by verified evidence. Return JSON: answer, confidence 0..1, status answered|abstain, supporting_doc_ids. If evidence remains insufficient return answer ABSTAIN and status abstain. No explanation.'''
def aggregate_answer(q:QueryInput,verification:dict,synth:dict,sees_question=True)->dict:
    assert_safe_query(q); payload={"verification":verification,"syntheses":synth}
    if sees_question:payload["question"]=q.question
    x=llm_json(AGG_PROMPT,payload); x["confidence"]=clip01(x.get("confidence"));
    if x.get("status")!="answered" or not str(x.get("answer","")).strip():x.update(answer="ABSTAIN",status="abstain")
    return x
def merge_evidence(old,new):
    out={}
    for m in ["text","table","image"]:
        vals={d["parent_doc_id"]:d for d in old.get(m,[])}
        for d in new.get(m,[]): vals.setdefault(d["parent_doc_id"],d)
        out[m]=list(vals.values())
    return out
def run_pipeline(q:QueryInput,cfg:dict)->dict:
    assert_safe_query(q); start=time.perf_counter(); plan=fallback_plan(q.question) if not cfg["planner"] else plan_query(q)
    evidence=retrieve_evidence(q,plan,cfg["graph"],fixed_all=not cfg["router"]); rounds=1; retry_reason=""; added=[]
    experts=run_modality_experts(q,evidence); synth=run_cross_modal_agents(q,experts,evidence)
    verification=verify_answer(q,synth,evidence) if cfg["verifier"] else {"status":"supported","best_answer":"","confidence":0.5,"supporting_doc_ids":[],"retry_modality":"none"}
    if cfg["feedback"] and verification.get("status") in {"conflict","insufficient"} and rounds<MAX_RETRIEVAL_ROUNDS:
        mod=verification.get("retry_modality"); rq=str(verification.get("retry_query","")).strip(); retry_reason=verification.get("status","")
        if mod in {"text","table","image"} and rq:
            rp={"subqueries":[rq],"needed_modalities":[mod],"needs_relational_search":cfg["graph"]}; new=retrieve_evidence(q,rp,cfg["graph"])
            before={d["parent_doc_id"] for xs in evidence.values() for d in xs}; evidence=merge_evidence(evidence,new); after={d["parent_doc_id"] for xs in evidence.values() for d in xs}; added=sorted(after-before); rounds=2
            experts=run_modality_experts(q,evidence); synth=run_cross_modal_agents(q,experts,evidence); verification=verify_answer(q,synth,evidence)
    if not cfg["verifier"]:
        verification={"status":"supported","best_answer":"","confidence":0.5,"supporting_doc_ids":[]}
    final=aggregate_answer(q,verification,synth,cfg["agg_question"])
    if cfg["verifier"] and verification.get("status")!="supported":final={"answer":"ABSTAIN","confidence":verification.get("confidence",0),"status":"abstain","supporting_doc_ids":verification.get("supporting_doc_ids",[])}
    return {"qid":q.qid,"question":q.question,"planner":plan,"retrieved":evidence,"stage1":experts,"cross_modal":synth,"verification":verification,"retrieval_rounds":rounds,"retry_reason":retry_reason,"new_docs_added":added,"final_answer":final["answer"],"confidence":final["confidence"],"abstain":final["status"]=="abstain","latency_seconds":time.perf_counter()-start}

In [ ]:
# Cells 25-26 - isolated oracle and experiment configurations
CONFIGS={
 "B1_Dense":{"planner":False,"router":False,"graph":False,"verifier":False,"feedback":False,"agg_question":True},
 "B2_Planner_Dense":{"planner":True,"router":True,"graph":False,"verifier":False,"feedback":False,"agg_question":True},
 "B3_Graph":{"planner":True,"router":True,"graph":True,"verifier":False,"feedback":False,"agg_question":True},
 "B4_Verifier":{"planner":True,"router":True,"graph":True,"verifier":True,"feedback":False,"agg_question":True},
 "FULL":{"planner":True,"router":True,"graph":True,"verifier":True,"feedback":True,"agg_question":True},
 "FULL_minus_graph":{"planner":True,"router":True,"graph":False,"verifier":True,"feedback":True,"agg_question":True},
 "FULL_minus_feedback":{"planner":True,"router":True,"graph":True,"verifier":True,"feedback":False,"agg_question":True},
 "FULL_minus_verifier":{"planner":True,"router":True,"graph":True,"verifier":False,"feedback":True,"agg_question":True},
 "FULL_minus_router":{"planner":True,"router":False,"graph":True,"verifier":True,"feedback":True,"agg_question":True},
 "FULL_agg_no_question":{"planner":True,"router":True,"graph":True,"verifier":True,"feedback":True,"agg_question":False}}
def oracle_pipeline(q:QueryInput,ev:EvaluationOnly)->dict:
    assert_safe_query(q); start=time.perf_counter(); evidence={m:[] for m in ["text","table","image"]}
    for did in ev.gold_support_ids:
        mod=ev.gold_support_modalities.get(did)
        rec=PARENT_LOOKUP.get((mod,did)) if mod else next((r for (m,i),r in PARENT_LOOKUP.items() if i==did),None)
        if rec and (mod!="image" or rec.get("image_path")):evidence[rec["modality"]].append({**rec,"dense_score":1,"graph_score":0,"final_score":1,"subqueries":["ORACLE"]})
    experts=run_modality_experts(q,evidence); synth=run_cross_modal_agents(q,experts,evidence); ver=verify_answer(q,synth,evidence); final=aggregate_answer(q,ver,synth,AGGREGATOR_SEES_QUESTION)
    return {"qid":q.qid,"question":q.question,"planner":{"oracle":True},"retrieved":evidence,"stage1":experts,"cross_modal":synth,"verification":ver,"retrieval_rounds":1,"retry_reason":"","new_docs_added":[],"final_answer":final["answer"],"confidence":final["confidence"],"abstain":final["status"]=="abstain","latency_seconds":time.perf_counter()-start}

In [ ]:
# Cells 27-28 - gold parsing, retrieval metrics and all-answer QA metrics
def answer_strings(row):
    vals=[]
    def walk(x):
        if isinstance(x,str): vals.append(x)
        elif isinstance(x,list):
            for y in x:walk(y)
        elif isinstance(x,dict):
            for k in ["answer","answers","aliases","normalized_answer"]:
                if k in x:walk(x[k])
    walk(row.get("answers",[])); return list(dict.fromkeys(v.strip() for v in vals if v.strip()))
def support_info(row):
    ids=[]; mods={}; known={m:{str(x["id"]) for x in raw} for m,raw in [("text",texts_raw),("table",tables_raw),("image",images_raw)]}
    def add(v,mod=None):
        s=str(v)
        if s in {"","None"}:return
        inferred=mod or next((m for m,z in known.items() if s in z),None)
        if inferred and s in known[inferred]:ids.append(s); mods[s]=inferred
    def walk(x,mod=None):
        if isinstance(x,list):
            for y in x:walk(y,mod)
        elif isinstance(x,dict):
            local=x.get("modality",x.get("doc_part",mod)); local={"text":"text","table":"table","image":"image"}.get(str(local).lower(),mod)
            for k,v in x.items():
                km={"text_doc_id":"text","table_id":"table","image_doc_id":"image","doc_id":local}.get(k)
                if km:add(v,km)
                elif k not in {"answer","text","content"}:walk(v,local)
        elif mod:add(x,mod)
    walk(row.get("supporting_context",[])); return list(dict.fromkeys(ids)),mods
def evaluation_only(row):
    ids,mods=support_info(row); meta=row.get("metadata",{})
    return EvaluationOnly(answer_strings(row),ids,mods,str(meta.get("type","unknown")),list(meta.get("modalities",[])),row.get("intermediate_answers"))
ARTICLES={"a","an","the"}
def norm_answer(s):
    s=str(s).lower(); s="".join(" " if c in string.punctuation else c for c in s); return " ".join(w for w in s.split() if w not in ARTICLES)
def em_f1(pred,golds):
    p=norm_answer(pred).split(); best_em=0; best_f1=0.0
    for g0 in golds:
        g=norm_answer(g0).split(); best_em=max(best_em,int(p==g)); common=Counter(p)&Counter(g); n=sum(common.values())
        f=0 if not p or not g or n==0 else 2*(n/len(p))*(n/len(g))/((n/len(p))+(n/len(g))); best_f1=max(best_f1,f)
    return best_em,best_f1
def retrieved_parent_ids(result,k,mod=None):
    xs=result["retrieved"].get(mod,[]) if mod else [x for ys in result["retrieved"].values() for x in ys]
    return [x["parent_doc_id"] for x in sorted(xs,key=lambda z:z.get("final_score",0),reverse=True)[:k]]
def retrieval_scores(result,ev):
    out={}; gold=set(ev.gold_support_ids)
    for k in [1,3,5,10]:
        got=retrieved_parent_ids(result,k); out[f"recall@{k}"]=len(gold&set(got))/len(gold) if gold else math.nan; out[f"hit@{k}"]=int(bool(gold&set(got))) if gold else math.nan; out[f"all_support_recall@{k}"]=int(gold<=set(got)) if gold else math.nan
        for m in ["text","table","image"]:
            gm={i for i in gold if ev.gold_support_modalities.get(i)==m}; ids=set(retrieved_parent_ids(result,k,m)); out[f"{m}_recall@{k}"]=len(gm&ids)/len(gm) if gm else math.nan
    ranked=retrieved_parent_ids(result,10); out["mrr"]=next((1/(i+1) for i,x in enumerate(ranked) if x in gold),0.0) if gold else math.nan
    return out

In [ ]:
# Cell 29 - deterministic stratified run selection (never first N)
def select_rows(rows,mode):
    eligible=list(rows)
    if not FULL_IMAGE_BENCHMARK:
        eligible=[r for r in eligible if "image" not in set(evaluation_only(r).gold_support_modalities.values())]
        print(f"Text/table development mode: excluded {len(rows)-len(eligible)} image-support questions from benchmark selection.")
    if mode=="full":return eligible
    n=16 if mode=="smoke" else 250; strata=defaultdict(list)
    for r in eligible:
        ev=evaluation_only(r); combo="+".join(sorted(set(ev.gold_support_modalities.values()))) or "unknown"; strata[(ev.metadata_type,combo)].append(r)
    rng=random.Random(SEED)
    for v in strata.values():rng.shuffle(v)
    out=[]
    while len(out)<min(n,len(eligible)):
        progressed=False
        for key in sorted(strata,key=str):
            if strata[key] and len(out)<n:out.append(strata[key].pop()); progressed=True
        if not progressed:break
    return out
SELECTED=select_rows(dev_raw,RUN_MODE); print("selected",len(SELECTED),"of",len(dev_raw))

In [ ]:
# Cell 30 - preflight assertions
assert all(PATHS.values()); assert all(query_only(r).question for r in SELECTED)
for r in SELECTED[:3]:assert_safe_query(query_only(r))
if FULL_IMAGE_BENCHMARK:
    assert len(IMAGE_PATHS)>0
    sample=[d for d in BY_MODALITY["image"] if d.get("image_path")][:5]
    CLIP.encode_missing(sample)
    sample_messages=[{"role":"user","content":[*[{"type":"image","image":f'file://{d["image_path"]}'} for d in sample],{"type":"text","text":"Validate image inputs."}]}]
    pixel_images,_=process_vision_info(sample_messages)
    pixel_prompt=processor.apply_chat_template(sample_messages,tokenize=False,add_generation_prompt=True)
    pixel_batch=processor(text=[pixel_prompt],images=pixel_images,return_tensors="pt")
    assert len(sample)>0 and pixel_images and "pixel_values" in pixel_batch
    print("IMAGE INPUT COUNT =",len(sample))
else: print("Image benchmark metrics will be omitted; metadata-only retrieval never counts as full image retrieval.")
table_test=next((r for r in dev_raw if any(m=="table" for m in evaluation_only(r).gold_support_modalities.values())),None)
if table_test:
    tq=query_only(table_test); table_hits=retrieve_evidence(tq,{"subqueries":[tq.question],"needed_modalities":["table"],"needs_relational_search":False},use_graph=False)["table"]
    tev=evaluation_only(table_test); print("TABLE SANITY gold/retrieved/hit:",tev.gold_support_ids,[x["parent_doc_id"] for x in table_hits],bool(set(tev.gold_support_ids)&{x["parent_doc_id"] for x in table_hits}))
print("LEAKAGE CHECK PASSED")

In [ ]:
# Cells 31-33 - resumable smoke/medium/full runs and optional ablations
def append_jsonl(path,row):
    with path.open("a",encoding="utf-8") as f:f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n"); f.flush(); os.fsync(f.fileno())
def completed(path):
    if not path.exists():return set()
    with path.open(encoding="utf-8") as f:return {json.loads(x)["qid"] for x in f if x.strip()}
def serializable_result(result):
    z=dict(result); z["retrieved"]={m:[{k:v for k,v in d.items() if k not in {"source_record"}} for d in xs] for m,xs in result["retrieved"].items()}; return z
def run_method(name,rows):
    path=DIRS["predictions"]/f"{name}.jsonl"; done=completed(path)
    for raw in rows:
        q=query_only(raw)
        if q.qid in done:continue
        # Gold is materialized only after the model/retriever has returned.
        if name=="MAMMQA_ORACLE":
            ev=evaluation_only(raw); result=oracle_pipeline(q,ev)
        else:
            result=run_pipeline(q,CONFIGS[name]); ev=evaluation_only(raw)
        em,f1=em_f1(result["final_answer"],ev.gold_answers); result=serializable_result(result)
        result.update(gold_answers=ev.gold_answers,gold_support_ids=ev.gold_support_ids,gold_support_modalities=ev.gold_support_modalities,metadata_type=ev.metadata_type,gold_modalities=ev.gold_modalities,exact_match=em,token_f1=f1,retrieval_metrics=retrieval_scores(result,ev))
        append_jsonl(path,result)
    return path
methods=["MAMMQA_ORACLE","B1_Dense","B2_Planner_Dense","FULL"] if RUN_MODE=="smoke" else ["MAMMQA_ORACLE","B1_Dense","B2_Planner_Dense","B3_Graph","B4_Verifier","FULL"]
RESULT_PATHS={name:run_method(name,SELECTED) for name in methods}
# Expensive ablations are opt-in and never run in smoke mode.
if RUN_MODE!="smoke" and RUN_ABLATIONS:
    ABLATION_PATHS={name:run_method(name,SELECTED) for name in CONFIGS if name.startswith("FULL_")}

In [ ]:
# Cells 34-35 - reports, error analysis and required exports
def read_jsonl(path):
    if not path.exists():return []
    with path.open(encoding="utf-8") as f:return [json.loads(x) for x in f if x.strip()]
def mean_valid(vals):
    v=[x for x in vals if x is not None and not (isinstance(x,float) and math.isnan(x))]; return float(np.mean(v)) if v else math.nan
def build_reports(result_paths:Dict[str,Path]):
    summary=[]; full_rows=[]
    for name,path in result_paths.items():
        rows=read_jsonl(path); r5=1.0 if name=="MAMMQA_ORACLE" else mean_valid([r["retrieval_metrics"].get("recall@5") for r in rows])
        summary.append({"Method":name,"Retrieval R@5":r5,"EM":mean_valid([r["exact_match"] for r in rows]),"F1":mean_valid([r["token_f1"] for r in rows]),"Abstention rate":mean_valid([r["abstain"] for r in rows])})
        if name=="FULL":full_rows=rows
    sdf=pd.DataFrame(summary); sdf.to_csv(DIRS["metrics"]/"ablation_results.csv",index=False)
    if full_rows:
        json.dump({k:mean_valid([r["retrieval_metrics"].get(k) for r in full_rows]) for k in full_rows[0]["retrieval_metrics"]},open(DIRS["metrics"]/"retrieval_metrics.json","w"),indent=2)
        json.dump({"EM":mean_valid([r["exact_match"] for r in full_rows]),"F1":mean_valid([r["token_f1"] for r in full_rows]),"abstention_rate":mean_valid([r["abstain"] for r in full_rows])},open(DIRS["metrics"]/"qa_metrics.json","w"),indent=2)
        pd.DataFrame([{"qid":r["qid"],"latency_seconds":r["latency_seconds"],"retrieval_rounds":r["retrieval_rounds"]} for r in full_rows]).to_csv(DIRS["metrics"]/"latency_results.csv",index=False)
        groups=defaultdict(list)
        for r in full_rows:
            groups[("question_type",r.get("metadata_type","unknown"))].append(r)
            combo="+".join(sorted(set(r.get("gold_support_modalities",{}).values()))) or "unknown"
            groups[("gold_modality_combination",combo)].append(r)
        pd.DataFrame([{"dimension":dim,"group":g,"n":len(rs),"EM":mean_valid([x["exact_match"] for x in rs]),"F1":mean_valid([x["token_f1"] for x in rs])} for (dim,g),rs in groups.items()]).to_csv(DIRS["metrics"]/"per_type_metrics.csv",index=False)
        errors=[]
        for r in full_rows:
            if r["exact_match"]:continue
            gold=set(r["gold_support_ids"]); got=set(retrieved_parent_ids(r,10)); upstream=any(norm_answer(x.get("candidate_answer","")) in {norm_answer(g) for g in r["gold_answers"]} for x in r.get("cross_modal",{}).values())
            if not gold<=got:kind="retrieval_failure"
            elif r["abstain"]:kind="abstention_failure"
            elif upstream:kind="aggregation_failure"
            elif any(r.get("gold_support_modalities",{}).get(i)=="image" for i in gold):kind="vision_failure"
            else:kind="reasoning_failure"
            errors.append({"qid":r["qid"],"error_type":kind,"prediction":r["final_answer"],"gold_answers":" | ".join(r["gold_answers"])})
        pd.DataFrame(errors).to_csv(DIRS["metrics"]/"error_analysis.csv",index=False)
        # Canonical requested aliases.
        (OUTPUT_ROOT/"predictions.jsonl").write_text((result_paths["FULL"]).read_text(encoding="utf-8"),encoding="utf-8")
        with (OUTPUT_ROOT/"retrieval_logs.jsonl").open("w",encoding="utf-8") as f:
            for r in full_rows:f.write(json.dumps({k:r[k] for k in ["qid","planner","retrieved","retrieval_rounds","retry_reason","new_docs_added"]},ensure_ascii=False)+"\n")
    config={k:v for k,v in globals().items() if k in {"SEED","RUN_MODE","MODEL_NAME","EMBED_MODEL","CLIP_MODEL","TEXT_TOP_K","TABLE_TOP_K","IMAGE_TOP_K","MAX_RETRIEVAL_ROUNDS","ENABLE_GRAPH","ENABLE_WEB","IMAGE_RETRIEVAL_MODE","AGGREGATOR_SEES_QUESTION","FULL_IMAGE_BENCHMARK"}}
    (OUTPUT_ROOT/"experiment_config.json").write_text(json.dumps(config,indent=2))
    print(sdf.to_string(index=False))
    if {"MAMMQA_ORACLE","FULL"}<=set(sdf.Method):
        a=sdf.set_index("Method"); print("Oracle-Full EM gap:",a.loc["MAMMQA_ORACLE","EM"]-a.loc["FULL","EM"],"F1 gap:",a.loc["MAMMQA_ORACLE","F1"]-a.loc["FULL","F1"])
    return sdf
summary=build_reports(RESULT_PATHS)